In [1]:
import os
os.environ['HF_ENDPOINT'] = 'https://hf-mirror.com'
os.environ.pop('http_proxy', None)
os.environ.pop('https_proxy', None)
os.environ.pop('HF_HUB_OFFLINE', None)
os.environ.pop('TRANSFORMERS_OFFLINE', None)
print("环境变量设置完成")

环境变量设置完成


In [2]:
import sys
sys.path.append('../scripts')

from importlib import import_module

retrieval_module = import_module('07_retrieval_pipeline')
generation_module = import_module('11_generation_pipeline')
test_module = import_module('12_test_generation_pipeline')

retrieval_pipeline = retrieval_module.MedRAGPipeline(
    chunks_path='../data/processed/chunks.parquet',
    chroma_db_path='../data/processed/chroma_db'
)

generation_pipeline = generation_module.MedicalGenerationPipeline(
    llm_model_name="deepseek-r1:7b",
    ollama_base_url="http://localhost:11434",
    enable_evidence_evaluation=False,
    enable_critical_review=False,
    llm_timeout=600
)

TEST_QUERIES = test_module.TEST_QUERIES

test_module.run_batch_test(
    pipeline=generation_pipeline,
    queries=TEST_QUERIES,
    retrieval_pipeline=retrieval_pipeline,
    output_path='../reports/generation_test_log_lite_v2_macmini.jsonl',
    top_k_final=5
)

初始化 MultiPathRetriever...
加载chunk数据...
连接ChromaDB...
加载embedding模型...


config.json:   0%|          | 0.00/743 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/366 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

config.json:   0%|          | 0.00/680 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/3.07k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

[ContextAssembler] tokenizer 加载成功: deepseek-ai/DeepSeek-R1-Distill-Qwen-7B
[OK] 已连接Ollama，模型 'deepseek-r1:7b' 可用。

========== [1/5] What is the effect of metformin on cardiovascular disease? ==========
  answer长度: 2224字符
  stage_success: {'context_assembly': True, 'answer_generator': True}
  幻觉引用: False
  已写入 ../reports/generation_test_log_lite_v2_macmini.jsonl

========== [2/5] What are the differences between metformin and sulfonylureas in treating type 2 diabetes? ==========
  answer长度: 2320字符
  stage_success: {'context_assembly': True, 'answer_generator': True}
  幻觉引用: False
  已写入 ../reports/generation_test_log_lite_v2_macmini.jsonl

========== [3/5] What are the common side effects of metformin? ==========
  answer长度: 1142字符
  stage_success: {'context_assembly': True, 'answer_generator': True}
  幻觉引用: False
  已写入 ../reports/generation_test_log_lite_v2_macmini.jsonl

========== [4/5] What is the recommended dosage of metformin for elderly patients? ==========
  answer长度: 891字符
  st

In [3]:
import json

log_path = '../reports/generation_test_log_lite_v2_macmini.jsonl'

with open(log_path, 'r') as f:
    logs = [json.loads(line) for line in f]

total_retrieval = 0
total_generation = 0

for i, log in enumerate(logs):
    retrieval_time = log.get('retrieval_time_seconds') or 0
    metrics = log.get('generation_metrics') or {}
    gen_time = metrics.get('total_time_seconds') or 0
    
    total_retrieval += retrieval_time
    total_generation += gen_time
    
    print(f"[{i+1}] 检索: {retrieval_time}秒  生成: {gen_time}秒")

total_all = total_retrieval + total_generation
print(f"\n5条合计: {round(total_all, 1)}秒 (约{round(total_all/60, 1)}分钟)")

[1] 检索: 3.04秒  生成: 65.5秒
[2] 检索: 2.08秒  生成: 54.53秒
[3] 检索: 1.47秒  生成: 52.7秒
[4] 检索: 1.56秒  生成: 55.49秒
[5] 检索: 1.32秒  生成: 53.91秒

5条合计: 291.6秒 (约4.9分钟)
